# 03 - Treat Service Users Snapshots

## Purpose
Count people *actually using* aged care per SA3 per year, broken down by care type
and Home Care Package (HCP) level. This is the demand side of the supply/demand story.

## Input
- `data/raw/service_users_snapshot_SA3/` — 9 pre-aggregated government release files:
  - Residential care × 2023 / 2024 / 2025 (file suffix: `-4-residential-service-location`)
  - Home care (recipient location) × 2023 / 2024 / 2025 (file suffix: `-2-home-care-recipient-location`)

## Output
- `data/clean/service_users_by_sa3.csv` — SA3 × year with user counts by care type and HCP level

## Key context
- HCP Level 1/2 = low-to-moderate needs
- HCP Level 3/4 = high needs — proxy for residential care waitlist pressure
- Recipient location = where the person *lives* (not where service is delivered from)
- GEN data is point-in-time (30 June each year), not annual admissions counts

In [1]:
import pandas as pd
import os

RAW = '../../data/raw/service_users_snapshot_SA3'
OUT = '../../data/clean/service_users_by_sa3.csv'

In [2]:
# =============================================================================
# STEP 1: Load residential care files (30 June snapshots, SA3 level)
# =============================================================================
# Files ending in "-4-residential-service-location" contain one row per SA3
# with counts of permanent + respite residents as at 30 June each year.
# "Service location" = the SA3 where the facility sits (not where the resident came from).
#
# NOTE: Skip temp files (Excel lock files starting with ~$) to avoid PermissionError.

def valid_xlsx(f): return f.endswith('.xlsx') and not f.startswith('~$')

res_files = sorted([f for f in os.listdir(RAW)
                    if 'residential' in f and 'service-location' in f and valid_xlsx(f)])
print('Residential files:')
for f in res_files:
    print(f'  {f}')

res_frames = []
for fname in res_files:
    # Extract year from filename pattern: "30-June-YYYY"
    year = int(fname.split('June-')[1].split('-')[0])
    xl = pd.ExcelFile(f'{RAW}/{fname}')

    # The SA3 sheet is always named with 'SA3' â€” skip state/national summary tabs
    sa3_sheet = [s for s in xl.sheet_names if 'SA3' in s][0]
    df = pd.read_excel(f'{RAW}/{fname}', sheet_name=sa3_sheet, header=None, skiprows=3)
    df.columns = ['sa3_code', 'sa3_name', 'permanent', 'respite', 'total_residential']

    # Remove non-data rows (headers, totals, footnotes mixed into the SA3 block)
    df = df[pd.to_numeric(df['sa3_code'], errors='coerce').notna()].copy()
    for c in ['permanent', 'respite', 'total_residential']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['year'] = year
    res_frames.append(df)
    print(f'{year}: {len(df)} SA3 rows, {df["total_residential"].sum():,.0f} total residents')

res = pd.concat(res_frames, ignore_index=True)
print(f'\nResidential combined: {res.shape}')

Residential files:
  GEN-data-People-using-aged-care-by-region-30-June-2023-4-residential-care-(service-location).xlsx
  GEN-data-People-using-aged-care-by-region-30-June-2024-4-residential-care-(service-location).xlsx
  GEN-data-People-using-aged-care-by-region-30-June-2025-4-residential-care-(service-location).xlsx
2023: 323 SA3 rows, 192,231 total residents


2024: 323 SA3 rows, 198,362 total residents
2025: 323 SA3 rows, 203,624 total residents

Residential combined: (969, 6)


In [3]:
# =============================================================================
# STEP 2: Load home care files (recipient location, HCP level breakdown)
# =============================================================================
# Files ending in "-2-home-care-recipient-location" show where HCP recipients LIVE.
# This matters for access equity: a person in a remote SA3 may receive care from
# a provider headquartered in a nearby city, but their need is in that remote SA3.
#
# HCP Level breakdown is the key analytic here:
#   Level 1 = basic care needs (~$10k/year subsidy)
#   Level 2 = low care needs   (~$18k/year)
#   Level 3 = intermediate     (~$40k/year)
#   Level 4 = high care needs  (~$56k/year)
#
# INSIGHT: A SA3 with high Level 3/4 concentration but few residential facilities
# = people trapped at home who need institutional care but can't access it.

hc_files = sorted([f for f in os.listdir(RAW) if 'home-care' in f and 'recipient' in f])
print('Home care (recipient location) files:')
for f in hc_files:
    print(f'  {f}')

hc_frames = []
for fname in hc_files:
    year = int(fname.split('June-')[1].split('-')[0])
    xl = pd.ExcelFile(f'{RAW}/{fname}')
    sa3_sheet = [s for s in xl.sheet_names if 'SA3' in s][0]
    df = pd.read_excel(f'{RAW}/{fname}', sheet_name=sa3_sheet, header=None, skiprows=3)
    df.columns = ['sa3_code', 'sa3_name',
                  'hcp_level1', 'hcp_level2', 'hcp_level3', 'hcp_level4',
                  'total_homecare']
    df = df[pd.to_numeric(df['sa3_code'], errors='coerce').notna()].copy()
    for c in ['hcp_level1', 'hcp_level2', 'hcp_level3', 'hcp_level4', 'total_homecare']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['year'] = year
    hc_frames.append(df)
    print(f'{year}: {len(df)} SA3 rows, {df["total_homecare"].sum():,.0f} total HCP recipients')

hc = pd.concat(hc_frames, ignore_index=True)
print(f'\nHome care combined: {hc.shape}')

Home care (recipient location) files:
  GEN-data-People-using-aged-care-by-region-30-June-2023-2-home-care-(recipient-location).xlsx
  GEN-data-People-using-aged-care-by-region-30-June-2024-2-home-care-(recipient-location).xlsx
  GEN-data-People-using-aged-care-by-region-30-June-2025-2-home-care-(recipient-location).xlsx
2023: 335 SA3 rows, 257,687 total HCP recipients
2024: 335 SA3 rows, 274,820 total HCP recipients
2025: 335 SA3 rows, 291,435 total HCP recipients

Home care combined: (1005, 8)


In [4]:
# =============================================================================
# STEP 3: Join residential + home care â†’ unified access table
# =============================================================================

access = res[['sa3_code', 'sa3_name', 'year', 'permanent', 'respite', 'total_residential']].merge(
    hc[['sa3_code', 'year', 'hcp_level1', 'hcp_level2', 'hcp_level3', 'hcp_level4', 'total_homecare']],
    on=['sa3_code', 'year'], how='outer'
)

# Total users across both care types
access['total_users'] = access['total_residential'].fillna(0) + access['total_homecare'].fillna(0)

# High-needs HCP (Level 3+4) = proxy for people who need but can't access residential care
# This is the hidden demand indicator: high % here + few residential beds = pressure point
access['hcp_high_needs'] = access['hcp_level3'].fillna(0) + access['hcp_level4'].fillna(0)
access['pct_hcp_high']   = access['hcp_high_needs'] / access['total_homecare']

print(f'Access SA3 combined shape: {access.shape}')
print(f'Years: {sorted(access["year"].unique())}')
print(f'SA3 regions: {access["sa3_code"].nunique()}')

print('\n=== National totals by year ===')
national = access.groupby('year')[['total_residential', 'total_homecare', 'hcp_high_needs']].sum()
national['pct_high_needs'] = national['hcp_high_needs'] / national['total_homecare'] * 100
print(national.round(1).to_string())
print('\n>> Rising pct_high_needs = more people accumulating at home waiting for residential care')

Access SA3 combined shape: (1005, 14)
Years: [np.int64(2023), np.int64(2024), np.int64(2025)]
SA3 regions: 671

=== National totals by year ===
      total_residential  total_homecare  hcp_high_needs  pct_high_needs
year                                                                   
2023           192231.0          257687          140968            54.7
2024           198362.0          274820          148386            54.0
2025           203624.0          291435          172285            59.1

>> Rising pct_high_needs = more people accumulating at home waiting for residential care


In [5]:
# =============================================================================
# STEP 4: Save
# =============================================================================

access['sa3_code'] = access['sa3_code'].astype(str).str.strip()
access.to_csv(OUT, index=False)

print(f'Saved: {access.shape[0]:,} rows Ã— {access.shape[1]} columns â†’ {OUT}')
print(f'Years: {sorted(access["year"].unique())}')
print(f'SA3 regions: {access["sa3_code"].nunique()}')
print('\nColumns:', access.columns.tolist())
print('\nSample:')
print(access.head(5).to_string(index=False))

Saved: 1,005 rows Ã— 14 columns â†’ ../../data/clean/service_users_by_sa3.csv
Years: [np.int64(2023), np.int64(2024), np.int64(2025)]
SA3 regions: 659

Columns: ['sa3_code', 'sa3_name', 'year', 'permanent', 'respite', 'total_residential', 'hcp_level1', 'hcp_level2', 'hcp_level3', 'hcp_level4', 'total_homecare', 'total_users', 'hcp_high_needs', 'pct_hcp_high']

Sample:
sa3_code        sa3_name  year  permanent  respite  total_residential  hcp_level1  hcp_level2  hcp_level3  hcp_level4  total_homecare  total_users  hcp_high_needs  pct_hcp_high
 10102.0      Queanbeyan  2024      254.0      8.0              262.0          14         252         157          74             497        759.0             231      0.464789
 10102.0      Queanbeyan  2025      267.0      4.0              271.0          13         228         170          87             498        769.0             257      0.516064
 10103.0 Snowy Mountains  2024       77.0      0.0               77.0           6          71     